# 03_VisitSummary: do the bad visits stand out in `preliminary_visit_summary`?

Notebook 02 found visits where `single_visit_star` photometry disagrees strongly with the Monster (median offset 2-15 %, sigma_MAD 5-20 %), for example 2025051900119 (y), 2025052000091 (g) and 2025052900063 (z), next to clean visits of the same bands.

Here we read `preliminary_visit_summary` (stage 1, one row per detector) for the visits processed in notebook 02 and compare the quantities it holds (zero point, PSF size, sky level, astrometric residuals, number of PSF stars, ...) between clean and bad visits.

Input: `data_02_Step1VsMonster_allvisits/summary_per_visit.csv` and `summary_per_detector.csv`, written by notebook 02.

Run with the `lsst` kernel at CC-IN2P3. The column names of the summary must be checked on the first run.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SRC = Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rubincc import monster as mon

NB_TAG = '03_VisitSummary'
DATA_DIR = Path(f'data_{NB_TAG}')
FIG_DIR = Path(f'figs_{NB_TAG}')
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.labelsize'] = 'x-large'
plt.rcParams['axes.titlesize'] = 'x-large'
plt.rcParams['xtick.labelsize'] = 'x-large'
plt.rcParams['ytick.labelsize'] = 'x-large'


def savefig(fig, name):
    # every figure is saved as PDF and PNG
    for ext in ('pdf', 'png'):
        fig.savefig(FIG_DIR / f'{NB_TAG}_{name}.{ext}', bbox_inches='tight', dpi=150)

In [ ]:
REPO = 'main'
USER = os.getenv('USER')
RUN_COLLECTION = f'u/{USER}/LSSTCam/runcosmos01_20261026_tract9813patch50_RunCosmosDemo/stage1'

NB02_DIR = Path('data_02_Step1VsMonster_allvisits')
BAD_SIGMA_MAD = 0.03   # visits above this sigma_MAD(dmag) are called 'bad'

per_visit = pd.read_csv(NB02_DIR / 'summary_per_visit.csv')
per_visit['bad'] = per_visit['sigma_mad'] > BAD_SIGMA_MAD
print(per_visit[['visit', 'band', 'n', 'median_dmag', 'sigma_mad', 'bad']].round(4).to_string(index=False))

## 1. Read `preliminary_visit_summary`

In [ ]:
run_butler = mon.get_butler(REPO, collections=RUN_COLLECTION)

frames = []
failures = []
for visit in per_visit['visit']:
    try:
        vs = run_butler.get('preliminary_visit_summary', visit=int(visit), instrument='LSSTCam')
        df = mon.as_dataframe(vs)
        df['visit'] = int(visit)
        frames.append(df)
    except Exception as exc:
        failures.append((visit, repr(exc)))
print('failures:', failures[:5])

vs_all = pd.concat(frames, ignore_index=True)
print(vs_all.shape)
print(list(vs_all.columns))

In [ ]:
# detector column: the summary is an ExposureCatalog whose id is the detector id
det_col = 'detector_id' if 'detector_id' in vs_all.columns else 'id'
vs_all['detector'] = vs_all[det_col]

# numeric quantities of interest, kept only if present in this pipeline version
WANTED = ['zeroPoint', 'psfSigma', 'psfArea', 'skyBg', 'skyNoise', 'nPsfStar', 'astromOffsetMean',
          'astromOffsetStd', 'zenithDistance', 'effTime', 'magLim', 'expTime']
KEY = [c for c in WANTED if c in vs_all.columns]
print('available:', KEY)
print('missing  :', [c for c in WANTED if c not in vs_all.columns])
vs_all[['visit', 'detector'] + KEY].describe().T.round(4)

## 2. Clean versus bad visits

In [ ]:
# median over detectors of each quantity, per visit, next to the Monster comparison
summ = vs_all.groupby('visit')[KEY].median().reset_index()
tbl = per_visit.merge(summ, on='visit', how='left').sort_values(['band', 'visit']).reset_index(drop=True)
# the zero point is defined for counts, so it depends on the exposure time: normalise it to 30 s
# (check: the 15 s visits should have a zero point about 0.75 mag lower than 30 s visits of the same band)
extra = []
if 'zeroPoint' in KEY and 'expTime' in KEY:
    tbl['zeroPoint_30s'] = tbl['zeroPoint'] - 2.5 * np.log10(tbl['expTime'] / 30.0)
    clean_med = tbl[~tbl['bad']].groupby('band')['zeroPoint_30s'].median()
    tbl['dZP_vs_clean'] = tbl['zeroPoint_30s'] - tbl['band'].map(clean_med)
    extra = ['zeroPoint_30s', 'dZP_vs_clean']
tbl.to_csv(DATA_DIR / f'{NB_TAG}_per_visit_table.csv', index=False)
cols = (['visit', 'band', 'bad'] + [c for c in ('observation_reason',) if c in tbl.columns]
        + ['median_dmag', 'sigma_mad'] + KEY + extra)
print(tbl[cols].round(4).to_string(index=False))

In [ ]:
# the same quantities against the visit order, bad visits circled in red
plot_vars = [c for c in ('zeroPoint_30s', 'psfSigma', 'skyBg', 'nPsfStar', 'astromOffsetStd') if c in tbl.columns]
order = {v: i for i, v in enumerate(sorted(tbl['visit']))}
tbl['x'] = tbl['visit'].map(order)
bands = sorted(tbl['band'].unique())

fig, axes = plt.subplots(len(plot_vars), 1, figsize=(12, 3.2 * len(plot_vars)), squeeze=False, sharex=True)
for ax, var in zip(axes.ravel(), plot_vars):
    for band in bands:
        s = tbl[tbl['band'] == band]
        ax.plot(s['x'], s[var], 'o', label=band)
    b = tbl[tbl['bad']]
    ax.scatter(b['x'], b[var], s=260, facecolors='none', edgecolors='r', linewidths=2, label='bad')
    ax.set_ylabel(var)
axes.ravel()[0].legend(ncol=len(bands) + 1)
axes.ravel()[-1].set_xlabel('visit index (sorted)')
savefig(fig, 'visit_summary_vs_visit')

## 3. Detector level for the bad visits

Per-detector medians of dmag (notebook 02) next to the per-detector quantities of the summary.

In [ ]:
per_det = pd.read_csv(NB02_DIR / 'summary_per_detector.csv')
dtbl = per_det.merge(vs_all[['visit', 'detector'] + KEY], on=['visit', 'detector'], how='left')
dtbl = dtbl.merge(per_visit[['visit', 'bad']], on='visit', how='left')
dtbl.to_csv(DATA_DIR / f'{NB_TAG}_per_detector_table.csv', index=False)

bad_visits = per_visit.loc[per_visit['bad'], 'visit'].tolist()
show = ['visit', 'band', 'detector', 'sensor_type', 'n', 'median_dmag', 'sigma_mad'] + [c for c in KEY if c in ('zeroPoint', 'psfSigma', 'skyBg', 'nPsfStar', 'astromOffsetMean')]
print(dtbl[dtbl['visit'].isin(bad_visits)][show].round(4).to_string(index=False))

In [ ]:
# detector-level view: is the offset of a detector related to the zero point or PSF size of that detector?
xvars = [c for c in ('zeroPoint', 'psfSigma') if c in dtbl.columns]
fig, axes = plt.subplots(1, len(xvars), figsize=(8 * len(xvars), 6), squeeze=False)
for ax, var in zip(axes.ravel(), xvars):
    for band in bands:
        s = dtbl[(dtbl['band'] == band) & (~dtbl['bad'])]
        ax.plot(s[var], s['median_dmag'], 'o', label=f'{band} clean')
    b = dtbl[dtbl['bad']]
    ax.plot(b[var], b['median_dmag'], 'rx', ms=10, mew=2, label='bad visits')
    ax.axhline(0, color='k', lw=1)
    ax.set_xlabel(var)
    ax.set_ylabel('median(psfFlux mag - Monster mag)')
axes.ravel()[0].legend()
savefig(fig, 'dmag_vs_summary_quantities')